# 🔍 Heartbeat Detective: Solving the Mystery of ECG Signals

Welcome back, biosignal explorer! Ready to become a heart detective?

## What You'll Learn Today

In 45 minutes, you'll:
- Load and explore real-looking ECG signals
- Find R-peaks automatically (the main heartbeat markers)
- Calculate accurate heart rate from the data
- Detect irregular beats that might signal problems
- Learn basic peak detection algorithms

## Skills You'll Master

- **Peak detection** - finding important points in signals
- **Timing analysis** - measuring time between events
- **Anomaly detection** - spotting unusual patterns

**Ready to solve some heartbeat mysteries? Let's go!** 🕵️‍♀️

## Step 1: Load Your Detective Tools

Every detective needs good tools! Let's get ours ready.

In [ ]:
# Import our detective toolkit
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Make our plots look professional
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Detective tools loaded! Ready to investigate heartbeats!")

---
# 📊 Understanding ECG Signals

## What is an ECG?

An **ECG** (electrocardiogram) is like a recording of your heart's electrical activity. Each heartbeat creates a specific pattern called the **PQRST complex**:

- **P wave**: Small bump when the top chambers squeeze
- **QRS complex**: Big spike when the main chambers pump blood (this is the **R-peak**!)
- **T wave**: Medium bump when the heart relaxes

## Why is the R-Peak Important?

The R-peak is the tallest, sharpest part of each heartbeat. Finding all the R-peaks tells us:
- Exactly when each heartbeat happens
- How fast the heart is beating
- If the rhythm is regular or irregular

Let's create a realistic ECG signal to practice on!

In [ ]:
def generate_realistic_ecg(duration, heart_rate, add_noise=True, irregular=False):
    """
    Generate a realistic ECG signal with customizable features
    
    duration: length in seconds
    heart_rate: average beats per minute
    add_noise: add realistic measurement noise
    irregular: make some beats irregular (like real hearts!)
    """
    sampling_rate = 500  # 500 samples per second
    time = np.linspace(0, duration, int(duration * sampling_rate))
    signal = np.zeros(len(time))
    
    beat_interval = 60 / heart_rate  # Average time between beats
    beat_times = []  # Track when each beat happens
    
    # Create each heartbeat
    current_time = 0.5  # Start at 0.5 seconds
    while current_time < duration - 1:
        beat_times.append(current_time)
        beat_index = int(current_time * sampling_rate)
        
        # Create the PQRST complex
        beat_length = int(0.6 * sampling_rate)
        if beat_index + beat_length < len(signal):
            t = np.linspace(0, 1, beat_length)
            
            # P wave (atrial contraction)
            p_wave = 0.25 * np.exp(-((t - 0.2) ** 2) / 0.005)
            
            # QRS complex (ventricular contraction) - the main event!
            q_wave = -0.1 * np.exp(-((t - 0.32) ** 2) / 0.0005)
            r_peak = 1.8 * np.exp(-((t - 0.35) ** 2) / 0.0008)
            s_wave = -0.15 * np.exp(-((t - 0.38) ** 2) / 0.0005)
            
            # T wave (ventricular recovery)
            t_wave = 0.35 * np.exp(-((t - 0.6) ** 2) / 0.01)
            
            beat = p_wave + q_wave + r_peak + s_wave + t_wave
            signal[beat_index:beat_index + beat_length] += beat
        
        # Calculate next beat time
        if irregular and np.random.random() < 0.15:  # 15% irregular
            # Make this beat come early or late
            variation = beat_interval * np.random.uniform(-0.3, 0.4)
        else:
            # Normal small variation
            variation = beat_interval * np.random.uniform(-0.05, 0.05)
        
        current_time += beat_interval + variation
    
    # Add realistic noise
    if add_noise:
        # Baseline wander (breathing and movement)
        baseline = 0.1 * np.sin(2 * np.pi * 0.3 * time)
        # High-frequency noise (muscle activity)
        noise = np.random.normal(0, 0.05, len(signal))
        signal = signal + baseline + noise
    
    return time, signal, beat_times

print("✅ ECG generator ready!")

## Let's Create Our First ECG Signal

We'll generate 10 seconds of ECG data from a healthy resting heart.

In [ ]:
# Generate a 10-second ECG
duration = 10  # seconds
heart_rate = 72  # beats per minute (normal resting)

time, ecg_signal, true_beats = generate_realistic_ecg(duration, heart_rate, add_noise=True)

# Plot the full signal
plt.figure(figsize=(15, 5))
plt.plot(time, ecg_signal, linewidth=1.5, color='darkblue', label='ECG Signal')
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Voltage (mV)', fontsize=12)
plt.title('🫀 Raw ECG Signal - Your Detective Case!', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"📊 Signal details:")
print(f"   Duration: {duration} seconds")
print(f"   Expected heart rate: {heart_rate} BPM")
print(f"   Expected beats: ~{int(heart_rate * duration / 60)}")
print(f"\n🔍 Your mission: Find all the R-peaks in this signal!")

## 🔎 Let's Zoom In

Let's look at just 3 seconds so we can see the PQRST complex clearly!

In [ ]:
# Show just the first 3 seconds in detail
zoom_end = 3  # seconds
zoom_mask = time <= zoom_end

plt.figure(figsize=(15, 6))
plt.plot(time[zoom_mask], ecg_signal[zoom_mask], linewidth=2, color='crimson')
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Voltage (mV)', fontsize=12)
plt.title('🔍 Zoomed View - Can You See the PQRST Patterns?', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add annotations to help identify parts
plt.text(0.7, 1.3, 'R-Peak\n(Find these!)', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
plt.show()

print("👀 Look for the tall, sharp spikes - those are R-peaks!")
print("   Each R-peak = one heartbeat")

---
# 🎯 Mission 1: Find All R-Peaks

## How Peak Detection Works

Finding peaks automatically is like teaching a computer to do what your eyes do naturally. We use these clues:

1. **Height**: R-peaks are taller than other parts (usually above 1.0 mV)
2. **Distance**: Hearts can't beat super fast, so peaks are at least 0.4 seconds apart
3. **Prominence**: R-peaks stick out clearly from surrounding signal

## The `find_peaks` Function

Python's `scipy.signal.find_peaks` function is our main tool. We give it:
- The signal to search
- A minimum **height** (how tall peaks must be)
- A minimum **distance** between peaks (in samples)

Let's try it!

In [ ]:
# Peak detection settings
sampling_rate = 500  # We use 500 samples per second
min_height = 0.8  # R-peaks should be at least 0.8 mV tall
min_distance = int(0.4 * sampling_rate)  # At least 0.4 seconds between beats

# Find the peaks!
peaks, properties = find_peaks(ecg_signal, height=min_height, distance=min_distance)

# Convert peak indices to times
peak_times = time[peaks]

# Show the results
plt.figure(figsize=(15, 6))
plt.plot(time, ecg_signal, linewidth=1.5, color='darkblue', label='ECG Signal', alpha=0.7)
plt.plot(peak_times, ecg_signal[peaks], 'ro', markersize=10, 
         label=f'Detected R-Peaks ({len(peaks)})', zorder=5)
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Voltage (mV)', fontsize=12)
plt.title('🎯 Peak Detection Results - Found the Heartbeats!', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.show()

print(f"🎉 SUCCESS! Found {len(peaks)} R-peaks")
print(f"\n📊 Detection details:")
print(f"   Minimum height threshold: {min_height} mV")
print(f"   Minimum distance: {min_distance} samples ({min_distance/sampling_rate:.2f} seconds)")
print(f"   Actual detected beats: {len(peaks)}")
print(f"   Expected beats: ~{int(heart_rate * duration / 60)}")

## 💡 Try This!

What happens if we set the height threshold too high or too low?

**Your task**: Go back to the cell above and try:
1. `min_height = 0.3` (too low - finds too many peaks)
2. `min_height = 2.0` (too high - misses real peaks)
3. Find the best value that finds all the R-peaks!

The right value should find about 12 peaks for 10 seconds at 72 BPM.

---
# 💓 Mission 2: Calculate Accurate Heart Rate

## How to Calculate Heart Rate

Once we have all the R-peaks, calculating heart rate is simple!

**Method 1: Count and Convert**
- Count total beats in the recording
- Divide by duration (in minutes)
- Result = beats per minute (BPM)

**Method 2: Average RR Intervals**
- Measure time between each consecutive beat (called RR intervals)
- Calculate the average interval
- Convert to BPM: `60 / average_interval`

Let's use both methods!

In [ ]:
# Method 1: Simple count
num_beats = len(peaks)
duration_minutes = duration / 60
heart_rate_method1 = num_beats / duration_minutes

# Method 2: RR interval average
rr_intervals = np.diff(peak_times)  # Time between consecutive peaks
average_rr = np.mean(rr_intervals)
heart_rate_method2 = 60 / average_rr

# Display results
print("💓 HEART RATE ANALYSIS")
print("="*50)
print(f"\nMethod 1 (Simple Count):")
print(f"   Beats detected: {num_beats}")
print(f"   Duration: {duration} seconds ({duration_minutes:.2f} minutes)")
print(f"   Heart Rate: {heart_rate_method1:.1f} BPM")

print(f"\nMethod 2 (RR Interval Average):")
print(f"   Average RR interval: {average_rr:.3f} seconds")
print(f"   Heart Rate: {heart_rate_method2:.1f} BPM")

print(f"\n✅ Both methods agree! Heart rate ≈ {heart_rate_method1:.0f} BPM")
print(f"\n🎯 Compare to target: {heart_rate} BPM")
error = abs(heart_rate_method1 - heart_rate)
print(f"   Estimation error: {error:.1f} BPM ({error/heart_rate*100:.1f}%)")

## 📊 Visualize RR Intervals

RR intervals tell us about heart rhythm. In a healthy heart, they should be fairly consistent!

In [ ]:
# Plot RR intervals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))

# RR intervals over time
ax1.plot(peak_times[1:], rr_intervals, 'o-', color='green', markersize=8, linewidth=2)
ax1.axhline(y=average_rr, color='red', linestyle='--', linewidth=2, label=f'Average: {average_rr:.3f} s')
ax1.set_xlabel('Time (seconds)', fontsize=11)
ax1.set_ylabel('RR Interval (seconds)', fontsize=11)
ax1.set_title('⏱️ Time Between Heartbeats (RR Intervals)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Histogram of RR intervals
ax2.hist(rr_intervals, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
ax2.axvline(x=average_rr, color='red', linestyle='--', linewidth=2, label=f'Average: {average_rr:.3f} s')
ax2.set_xlabel('RR Interval (seconds)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('📊 Distribution of RR Intervals', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Calculate variability
rr_std = np.std(rr_intervals)
rr_variability = (rr_std / average_rr) * 100

print(f"\n📈 Heart Rate Variability (HRV):")
print(f"   Standard deviation: {rr_std:.3f} seconds")
print(f"   Variability: {rr_variability:.1f}%")
print(f"\n💡 Small variations are normal and healthy!")
print(f"   Too much or too little variability can indicate health issues.")

---
# ⚠️ Mission 3: Detect Irregular Beats

## What Are Irregular Beats?

Sometimes hearts skip a beat or add an extra beat. This is called an **arrhythmia**. We can detect irregular beats by looking for RR intervals that are very different from the average.

## Detection Method

An RR interval is considered irregular if:
- It's **much shorter** than average (premature beat)
- It's **much longer** than average (missed beat or pause)

We'll flag any interval that's more than 20% different from the average.

Let's create a signal WITH irregular beats and detect them!

In [ ]:
# Generate ECG with irregular beats
print("🔧 Generating ECG with some irregular beats...\n")
time_irr, ecg_irr, true_beats_irr = generate_realistic_ecg(
    duration=15, 
    heart_rate=75, 
    add_noise=True,
    irregular=True  # Add irregular beats!
)

# Detect peaks
peaks_irr, _ = find_peaks(ecg_irr, height=0.8, distance=int(0.4 * sampling_rate))
peak_times_irr = time_irr[peaks_irr]
rr_intervals_irr = np.diff(peak_times_irr)

# Find irregular intervals
mean_rr = np.mean(rr_intervals_irr)
threshold = 0.20  # 20% deviation

irregular_mask = np.abs(rr_intervals_irr - mean_rr) > (threshold * mean_rr)
irregular_indices = np.where(irregular_mask)[0]

print(f"✅ Detected {len(peaks_irr)} total heartbeats")
print(f"⚠️  Found {len(irregular_indices)} irregular beats ({len(irregular_indices)/len(rr_intervals_irr)*100:.1f}%)")

In [ ]:
# Visualize irregular beats
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Plot ECG with irregular beats highlighted
ax1.plot(time_irr, ecg_irr, linewidth=1.5, color='darkblue', alpha=0.6, label='ECG Signal')
ax1.plot(peak_times_irr, ecg_irr[peaks_irr], 'go', markersize=8, label='Normal Beats', alpha=0.6)

# Highlight irregular beats
for idx in irregular_indices:
    # Mark the beat AFTER the irregular interval
    ax1.plot(peak_times_irr[idx+1], ecg_irr[peaks_irr[idx+1]], 'r*', 
             markersize=20, markeredgecolor='darkred', markeredgewidth=2)

ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_ylabel('Voltage (mV)', fontsize=12)
ax1.set_title('⚠️ ECG with Irregular Beats Highlighted (Red Stars)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Plot RR intervals with irregular ones highlighted
colors = ['red' if irregular else 'green' for irregular in irregular_mask]
ax2.scatter(peak_times_irr[1:], rr_intervals_irr, c=colors, s=100, alpha=0.7, edgecolors='black')
ax2.axhline(y=mean_rr, color='blue', linestyle='--', linewidth=2, label=f'Mean RR: {mean_rr:.3f} s')
ax2.axhline(y=mean_rr * (1 + threshold), color='orange', linestyle=':', linewidth=2, label='Normal Range')
ax2.axhline(y=mean_rr * (1 - threshold), color='orange', linestyle=':', linewidth=2)
ax2.fill_between(time_irr, mean_rr * (1 - threshold), mean_rr * (1 + threshold), 
                 color='green', alpha=0.1, label='Normal Zone')
ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_ylabel('RR Interval (seconds)', fontsize=12)
ax2.set_title('📊 RR Intervals - Red = Irregular, Green = Normal', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Report irregular beats
print("\n" + "="*60)
print("⚠️  IRREGULAR BEAT DETECTION REPORT")
print("="*60)
print(f"\nTotal beats analyzed: {len(rr_intervals_irr)}")
print(f"Irregular beats found: {len(irregular_indices)}")
print(f"\nDetails of irregular beats:")
for i, idx in enumerate(irregular_indices, 1):
    interval = rr_intervals_irr[idx]
    deviation = ((interval - mean_rr) / mean_rr) * 100
    beat_type = "Early" if interval < mean_rr else "Late"
    print(f"   {i}. Time: {peak_times_irr[idx+1]:.2f}s | RR: {interval:.3f}s | {beat_type} ({deviation:+.1f}%)")

## 💡 Try This!

Can you adjust the irregularity detection?

**Experiment**: Change the `threshold` value in the cell above:
- `threshold = 0.10` (10% - very sensitive, finds more irregularities)
- `threshold = 0.30` (30% - less sensitive, only finds major irregularities)
- Find the best threshold that catches real problems without too many false alarms!

---
# 🧪 Your Turn - Detective Challenges!

Time to put your skills to the test!

## Exercise 1: Athlete's Heart 🏃‍♀️

Athletes often have slower resting heart rates (around 50-60 BPM) because their hearts are so efficient.

**Your task**: 
1. Create a 20-second ECG with heart rate = 55 BPM
2. Detect the R-peaks
3. Calculate the heart rate
4. Verify your detection is accurate!

In [ ]:
# Exercise 1: Athlete ECG
# TODO: Fill in the values!

athlete_duration = 20  # seconds
athlete_hr = 55  # BPM

# Generate the signal
time_ath, ecg_ath, _ = generate_realistic_ecg(athlete_duration, athlete_hr)

# YOUR CODE HERE: Detect peaks
peaks_ath, _ = find_peaks(ecg_ath, height=0.8, distance=int(0.5 * sampling_rate))  # Hint: slower heart = more distance!

# YOUR CODE HERE: Calculate heart rate
detected_hr = len(peaks_ath) / (athlete_duration / 60)

# Visualize
plt.figure(figsize=(15, 5))
plt.plot(time_ath, ecg_ath, 'b-', alpha=0.7)
plt.plot(time_ath[peaks_ath], ecg_ath[peaks_ath], 'ro', markersize=10)
plt.title(f'🏃‍♀️ Athlete ECG - Detected: {detected_hr:.1f} BPM', fontsize=14, fontweight='bold')
plt.xlabel('Time (seconds)')
plt.ylabel('Voltage (mV)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Expected: {athlete_hr} BPM")
print(f"Detected: {detected_hr:.1f} BPM")
print(f"Error: {abs(detected_hr - athlete_hr):.1f} BPM")

## Exercise 2: Exercise Heart Rate 💪

During exercise, heart rate increases significantly (120-170 BPM).

**Your task**:
1. Create a 15-second ECG at 140 BPM
2. Detect peaks (careful with the distance parameter!)
3. Calculate average RR interval
4. How many milliseconds between beats?

In [ ]:
# Exercise 2: Exercise ECG
exercise_hr = 140  # BPM - running pace!

# YOUR CODE HERE: Generate and analyze
time_ex, ecg_ex, _ = generate_realistic_ecg(15, exercise_hr)

# Detect peaks (beats are closer together now!)
peaks_ex, _ = find_peaks(ecg_ex, height=0.8, distance=int(0.3 * sampling_rate))

# Calculate RR intervals
rr_ex = np.diff(time_ex[peaks_ex])
avg_rr_ms = np.mean(rr_ex) * 1000  # Convert to milliseconds

print(f"\n💪 Exercise ECG Analysis:")
print(f"   Heart Rate: {exercise_hr} BPM")
print(f"   Beats detected: {len(peaks_ex)}")
print(f"   Average RR interval: {np.mean(rr_ex):.3f} seconds")
print(f"   Average RR interval: {avg_rr_ms:.0f} milliseconds")
print(f"\n🎯 Fun fact: At {exercise_hr} BPM, your heart beats every {avg_rr_ms:.0f}ms!")

# Plot
plt.figure(figsize=(15, 5))
plt.plot(time_ex, ecg_ex, 'g-', alpha=0.7, linewidth=1.5)
plt.plot(time_ex[peaks_ex], ecg_ex[peaks_ex], 'ro', markersize=8)
plt.title(f'💪 Exercise ECG - {exercise_hr} BPM', fontsize=14, fontweight='bold')
plt.xlabel('Time (seconds)')
plt.ylabel('Voltage (mV)')
plt.grid(True, alpha=0.3)
plt.show()

## Exercise 3: Build a Heart Monitor Function 🏥

Create a complete function that analyzes any ECG signal!

**Your task**: Complete the function below that:
1. Detects R-peaks
2. Calculates heart rate
3. Finds irregular beats
4. Creates a summary report

In [ ]:
def analyze_ecg(time, signal, sampling_rate=500):
    """
    Complete ECG analysis function
    
    Returns: dictionary with all analysis results
    """
    # Detect R-peaks
    peaks, _ = find_peaks(signal, height=0.8, distance=int(0.4 * sampling_rate))
    peak_times = time[peaks]
    
    # Calculate heart rate
    duration_min = (time[-1] - time[0]) / 60
    heart_rate = len(peaks) / duration_min
    
    # Calculate RR intervals
    rr_intervals = np.diff(peak_times)
    avg_rr = np.mean(rr_intervals)
    
    # Detect irregular beats (>20% deviation)
    irregular_mask = np.abs(rr_intervals - avg_rr) > (0.2 * avg_rr)
    num_irregular = np.sum(irregular_mask)
    
    # Create results dictionary
    results = {
        'peaks': peaks,
        'num_beats': len(peaks),
        'heart_rate': heart_rate,
        'avg_rr_interval': avg_rr,
        'rr_variability': np.std(rr_intervals),
        'num_irregular': num_irregular,
        'irregular_percent': (num_irregular / len(rr_intervals)) * 100 if len(rr_intervals) > 0 else 0
    }
    
    return results

# Test it on our irregular ECG
results = analyze_ecg(time_irr, ecg_irr)

print("\n" + "="*60)
print("🏥 ECG ANALYSIS REPORT")
print("="*60)
print(f"\n💓 Heart Rate: {results['heart_rate']:.1f} BPM")
print(f"🫀 Total Beats: {results['num_beats']}")
print(f"⏱️  Average RR: {results['avg_rr_interval']:.3f} seconds")
print(f"📊 RR Variability: {results['rr_variability']:.3f} seconds")
print(f"⚠️  Irregular Beats: {results['num_irregular']} ({results['irregular_percent']:.1f}%)")

if results['irregular_percent'] > 10:
    print(f"\n🚨 WARNING: High irregularity detected! Medical consultation recommended.")
elif results['irregular_percent'] > 5:
    print(f"\n⚠️  CAUTION: Some irregularity detected. Monitor closely.")
else:
    print(f"\n✅ NORMAL: Rhythm appears regular and healthy.")

---
# 🏆 Bonus Challenge: Real-Time Heart Monitor

Create a simple heart monitor that analyzes multiple people!

**Your task**: Analyze ECG signals from 4 different scenarios and compare them.

In [ ]:
# Bonus: Analyze multiple scenarios
scenarios = {
    '😴 Sleeping': 55,
    '🧘 Relaxed': 70,
    '🚶 Walking': 95,
    '🏃 Running': 150
}

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
summary = []

for ax, (scenario, target_hr) in zip(axes, scenarios.items()):
    # Generate signal
    t, sig, _ = generate_realistic_ecg(10, target_hr, add_noise=True)
    
    # Analyze
    results = analyze_ecg(t, sig)
    
    # Plot
    ax.plot(t, sig, 'b-', alpha=0.6, linewidth=1.5)
    ax.plot(t[results['peaks']], sig[results['peaks']], 'ro', markersize=6)
    ax.set_ylabel('mV', fontsize=10)
    ax.set_title(f"{scenario} - {results['heart_rate']:.0f} BPM | {results['num_irregular']} irregular beats", 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 10)
    
    summary.append((scenario, target_hr, results['heart_rate'], results['num_irregular']))

axes[-1].set_xlabel('Time (seconds)', fontsize=11)
plt.tight_layout()
plt.show()

# Summary table
print("\n" + "="*70)
print("📊 MULTI-SCENARIO HEART MONITOR SUMMARY")
print("="*70)
print(f"{'Scenario':<20} {'Target HR':<12} {'Detected HR':<15} {'Irregular'}")
print("-"*70)
for scenario, target, detected, irregular in summary:
    print(f"{scenario:<20} {target:>6} BPM    {detected:>6.1f} BPM      {irregular:>3}")
print("="*70)

---
# 🎉 Congratulations, Heart Detective!

## What You Mastered Today

Amazing work! You now know how to:

### 🔍 Peak Detection Skills
- ✅ Find R-peaks automatically using algorithms
- ✅ Tune detection parameters (height, distance)
- ✅ Validate detection accuracy

### 💓 Heart Rate Analysis
- ✅ Calculate heart rate from R-peaks
- ✅ Measure RR intervals precisely
- ✅ Understand heart rate variability (HRV)

### ⚠️ Irregularity Detection
- ✅ Identify abnormal heartbeat timing
- ✅ Classify beats as early or late
- ✅ Calculate irregularity percentages

### 🛠️ Practical Skills
- ✅ Build complete ECG analysis functions
- ✅ Visualize results professionally
- ✅ Generate analysis reports

## 🚀 What's Next?

You're ready for more adventures!

**Next Notebook**: `03_Brain_Wave_Explorer.ipynb` - Explore EEG signals and brain wave analysis!

## 💡 Real-World Applications

The skills you learned today are used in:
- 🏥 Hospital heart monitors
- ⌚ Fitness trackers and smartwatches
- 🚑 Ambulance ECG machines
- 📱 Mobile health apps
- 🔬 Medical research

---

### 📝 Key Takeaways

> **Peak Detection**: Finding important points in signals using height and distance thresholds
>
> **RR Intervals**: Time between heartbeats - key to understanding heart rhythm
>
> **Irregularity**: Beats that deviate significantly from the normal pattern

**Keep detecting, keep learning!** 🔬✨

---

*Made with ❤️ for curious minds by the Delta-Predictive-Biosensing team*